In [9]:
import polars as pl

embedding_batches = pl.read_parquet(
    "../data/embedding_batches/",
    columns=["comment_id", "embedding"],
)

yt_cmt = pl.read_csv("../data/yt_cmt/yt_cmt.csv")
composed_data = yt_cmt.join(embedding_batches, how='inner', on='comment_id')

print(len(embedding_batches))
print(embedding_batches.schema)
print(composed_data.head())
print(composed_data.schema)

395634
Schema({'comment_id': String, 'embedding': Array(Float32, shape=(768,))})
shape: (5, 22)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ comment_i ┆ post_id   ┆ comment_t ┆ published ┆ … ┆ mention_c ┆ emoji_cou ┆ like_coun ┆ embeddin │
│ d         ┆ ---       ┆ ext       ┆ _at       ┆   ┆ ount      ┆ nt        ┆ t_log     ┆ g        │
│ ---       ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ str       ┆           ┆ str       ┆ str       ┆   ┆ i64       ┆ i64       ┆ f64       ┆ array[f3 │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ 2, 768]  │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ UgxVtm4hx ┆ R1WQVeCq0 ┆ Colombia  ┆ 2026-04-3 ┆ … ┆ 0         ┆ 1         ┆ 8.187577  ┆ [0.01369 │
│ y8sDtUnxv ┆ Hs        ┆ y Brazil  ┆ 0T21:08:2 ┆   ┆           ┆           ┆           ┆ 5, -0.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import polars as pl
import numpy as np

comment_text_df = yt_cmt.select(
    (pl.col("comment_id")).alias("comment_id"),
    (pl.col("comment_text")).alias("comment_text")
)
comment_text_df = comment_text_df.drop_nulls()
comment_text_df = comment_text_df.drop_nans()
comments = comment_text_df["comment_text"].to_list()
comment_ids = comment_text_df["comment_id"].to_list()


[[ 3.72630544e-02 -8.85638292e-04 -1.16354518e-03 ... -1.15958904e-03
  -1.75036695e-02 -9.06647276e-03]
 [ 2.83542573e-02  3.94710456e-04  5.99798805e-04 ...  3.22445552e-03
  -1.10562798e-02  6.88885106e-03]
 [ 2.77576111e-02  6.50646747e-04  1.49343093e-03 ...  4.27379971e-03
  -2.49453988e-02 -3.52871255e-03]
 ...
 [ 3.78265940e-02  3.71358497e-03  3.71195399e-03 ... -8.39075632e-03
   1.25198252e-03 -4.66840994e-03]
 [ 4.23767492e-02  9.76931334e-01 -9.91487652e-02 ... -1.25318929e-03
  -1.81768986e-03 -4.78865841e-05]
 [ 1.31484583e-01  2.99222052e-01 -7.84789398e-02 ...  4.43655346e-03
  -3.93501436e-03  1.44279795e-02]]


AttributeError: 'DataFrame' object has no attribute 'to_parquet'

In [17]:
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",      
    ngram_range=(3, 7),       
    max_features=25_000,       
    sublinear_tf=True,
)
char_tfidf = char_vectorizer.fit_transform(comments)

svd_char = TruncatedSVD(n_components=300, random_state=42)
char_dense = svd_char.fit_transform(char_tfidf).astype(np.float32)
print(char_dense)

embedding_df = pl.DataFrame({
    "comment_id": comment_ids,
    "embedding": pl.Series(
        "embedding",
        [list(row) for row in char_dense],
        dtype=pl.Array(pl.Float32, 300),
    ),
})
embedding_df.write_parquet("../data/37gram/37gram.parquet")
embedding_df.head(10)

[[ 3.72630544e-02 -8.85638292e-04 -1.16354518e-03 ... -1.15958904e-03
  -1.75036695e-02 -9.06647276e-03]
 [ 2.83542573e-02  3.94710456e-04  5.99798805e-04 ...  3.22445552e-03
  -1.10562798e-02  6.88885106e-03]
 [ 2.77576111e-02  6.50646747e-04  1.49343093e-03 ...  4.27379971e-03
  -2.49453988e-02 -3.52871255e-03]
 ...
 [ 3.78265940e-02  3.71358497e-03  3.71195399e-03 ... -8.39075632e-03
   1.25198252e-03 -4.66840994e-03]
 [ 4.23767492e-02  9.76931334e-01 -9.91487652e-02 ... -1.25318929e-03
  -1.81768986e-03 -4.78865841e-05]
 [ 1.31484583e-01  2.99222052e-01 -7.84789398e-02 ...  4.43655346e-03
  -3.93501436e-03  1.44279795e-02]]


comment_id,embedding
str,"array[f32, 300]"
"""UgxVtm4hxy8sDtUnxv94AaABAg""","[0.037263, -0.000886, … -0.009066]"
"""UgzNM-cNmiDziXizsRB4AaABAg""","[0.028354, 0.000395, … 0.006889]"
"""Ugyk1pkJ6dtfPaQBsZp4AaABAg""","[0.027758, 0.000651, … -0.003529]"
"""Ugw8riDxbs4mhPWXuA94AaABAg""","[0.051167, -0.001648, … -0.016129]"
"""UgyAGclKh3tZFJCbflR4AaABAg""","[0.058485, -0.004498, … -0.003424]"
"""UgyQMMzclSbS8dYw6TR4AaABAg""","[0.061425, -0.001041, … 0.008251]"
"""UgyP8DY4aTutxzt3tWh4AaABAg""","[0.042223, -0.000058, … 0.00138]"
"""Ugwyq2h_5V3MA-NH4cJ4AaABAg""","[0.153966, -0.005828, … 0.014674]"
"""UgzKoWivjlY67yg9KT94AaABAg""","[0.06812, 0.000436, … 0.001724]"


In [18]:
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 3),
    max_features=5_000,
    sublinear_tf=True,
    stop_words=["de", "a", "e", "o", "y", "que", "la", "el", "en", "the", "and"]
)
word_tfidf = word_vectorizer.fit_transform(comments)

svd_word = TruncatedSVD(n_components=200, random_state=42)
word_dense = svd_word.fit_transform(word_tfidf).astype(np.float32)

embedding_df13 = pl.DataFrame({
    "comment_id": comment_ids,
    "embedding": pl.Series(
        "embedding",
        [list(row) for row in char_dense],
        dtype=pl.Array(pl.Float32, 300),
    ),
})
embedding_df13.write_parquet("../data/13gram/13gram.parquet")
embedding_df13.head(10)

comment_id,embedding
str,"array[f32, 300]"
"""UgxVtm4hxy8sDtUnxv94AaABAg""","[0.037263, -0.000886, … -0.009066]"
"""UgzNM-cNmiDziXizsRB4AaABAg""","[0.028354, 0.000395, … 0.006889]"
"""Ugyk1pkJ6dtfPaQBsZp4AaABAg""","[0.027758, 0.000651, … -0.003529]"
"""Ugw8riDxbs4mhPWXuA94AaABAg""","[0.051167, -0.001648, … -0.016129]"
"""UgyAGclKh3tZFJCbflR4AaABAg""","[0.058485, -0.004498, … -0.003424]"
"""UgyQMMzclSbS8dYw6TR4AaABAg""","[0.061425, -0.001041, … 0.008251]"
"""UgyP8DY4aTutxzt3tWh4AaABAg""","[0.042223, -0.000058, … 0.00138]"
"""Ugwyq2h_5V3MA-NH4cJ4AaABAg""","[0.153966, -0.005828, … 0.014674]"
"""UgzKoWivjlY67yg9KT94AaABAg""","[0.06812, 0.000436, … 0.001724]"


In [20]:
import numpy as np
import polars as pl
from gensim.models import KeyedVectors

wv = KeyedVectors.load_word2vec_format(
    "/media/rocminfo/565A28A25A2880BB/assets/NMDS/resources/fasttext/crawl-300d-2M.vec",
    binary=False,
    limit=None
)

def comment_vec_average(comment: str, wv, dim=300):
    """Average of word vectors; OOV words are skipped."""
    words = comment.split()
    vecs = [wv[word] for word in words if word in wv]
    if len(vecs) == 0:
        return np.zeros(dim, dtype=np.float32)
    return np.mean(vecs, axis=0).astype(np.float32)

# Example: compute embeddings for your list of comments
# comments = df["comment_text"].to_list()
# comment_ids = df["comment_id"].to_list()
embeddings = [comment_vec_average(c, wv) for c in comments]

dim = len(embeddings[0])

embedding_df_ft = pl.DataFrame({
    "comment_id": comment_ids,
    "embedding": pl.Series(
        "embedding",
        [list(vec) for vec in embeddings],
        dtype=pl.Array(pl.Float32, dim)
    )
})

embedding_df_ft.write_parquet("../data/fasttext_avg/fasttext_300d.parquet")

print(embedding_df_ft.head(10))

shape: (10, 2)
┌────────────────────────────┬─────────────────────────────────┐
│ comment_id                 ┆ embedding                       │
│ ---                        ┆ ---                             │
│ str                        ┆ array[f32, 300]                 │
╞════════════════════════════╪═════════════════════════════════╡
│ UgxVtm4hxy8sDtUnxv94AaABAg ┆ [-0.0173, 0.36276, … 0.0944]    │
│ UgzNM-cNmiDziXizsRB4AaABAg ┆ [-0.085178, 0.208311, … -0.279… │
│ Ugyk1pkJ6dtfPaQBsZp4AaABAg ┆ [-0.085075, -0.044075, … 0.022… │
│ Ugw8riDxbs4mhPWXuA94AaABAg ┆ [-0.0262, 0.348057, … -0.23248… │
│ UgyAGclKh3tZFJCbflR4AaABAg ┆ [-0.070325, 0.31725, … 0.10305… │
│ UgyQMMzclSbS8dYw6TR4AaABAg ┆ [-0.0415, 0.37735, … -0.1982]   │
│ UgyP8DY4aTutxzt3tWh4AaABAg ┆ [-0.071371, 0.3332, … -0.0147]  │
│ Ugwyq2h_5V3MA-NH4cJ4AaABAg ┆ [-0.08195, -0.126, … -0.031225… │
│ UgzKoWivjlY67yg9KT94AaABAg ┆ [0.0603, 0.197856, … -0.059011… │
│ UgxP2uPT9P02W9f87jd4AaABAg ┆ [-0.130833, -0.020033, … 0.043… │
└─────────

# Combined Parquet

In [2]:
import polars as pl

# 1. Original composed data (yt_cmt + first embedding)
composed = pl.read_parquet("../data/embedding_batches/", columns=["comment_id", "embedding"])  # actually reading directory – adjust if needed
# But you already have composed_data from the join; we'll assume you can reuse it.
# For safety, we read the original yt_cmt and join with the embedding batches.
yt_cmt = pl.read_csv("../data/yt_cmt/yt_cmt.csv")
emb_batch = pl.read_parquet("../data/embedding_batches/", columns=["comment_id", "embedding"])
composed = yt_cmt.join(emb_batch, on="comment_id", how="inner")

# 2. Character n‑gram embeddings (already has comment_id and embedding column)
char_emb = pl.read_parquet("../data/37gram/37gram.parquet").rename({"embedding": "embedding_char"})

# 3. Word n‑gram embeddings (fix the column name)
word_emb = pl.read_parquet("../data/13gram/13gram.parquet").rename({"embedding": "embedding_word"})

# 4. FastText average embeddings
ft_emb = pl.read_parquet("../data/fasttext_avg/fasttext_300d.parquet").rename({"embedding": "embedding_ft"})

# Now join all together (inner join keeps only comments present in all sources)
combined = (
    composed
    .join(char_emb, on="comment_id", how="inner")
    .join(word_emb, on="comment_id", how="inner")
    .join(ft_emb, on="comment_id", how="inner")
)

# Save to a single Parquet file
combined.write_parquet("../data/combined_embeddings.parquet")

In [3]:
print(combined.schema)

Schema({'comment_id': String, 'post_id': String, 'comment_text': String, 'published_at': String, 'like_count': Int64, 'reply_count': Int64, 'author_id': String, 'author_name': String, 'title_youtube': String, 'source_query': String, 'crawled_at': String, 'char_count': Int64, 'word_count': Int64, 'avg_word_length': Float64, 'uppercase_ratio': Float64, 'exclamation_count': Int64, 'question_count': Int64, 'hashtag_count': Int64, 'mention_count': Int64, 'emoji_count': Int64, 'like_count_log': Float64, 'embedding': Array(Float32, shape=(768,)), 'embedding_char': Array(Float32, shape=(300,)), 'embedding_word': Array(Float32, shape=(300,)), 'embedding_ft': Array(Float32, shape=(300,))})


# Labeled with original data

In [ ]:
import polars as pl

combined = pl.read_parquet("../data/combined_embeddings.parquet")
combined_labeled = pl.read_parquet("../data/combined_labeled.parquet")
combined_labeled = combined_labeled.unique(subset="comment_id", keep="first", maintain_order=True)

print(f"Length of combined: {len(combined)}")
print(f"Length of combined with labels: {len(combined_labeled)}")
final_df = combined.join(combined_labeled, on='comment_id')
print(f"Length of final joined dataset: {len(final_df)}")

Length of combined: 395479
Length of combined with labels: 395633
Length of final joined dataset: 395479
